[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/15-projects/p04-sentiment-classic-vs-neural/colab_quickstart.ipynb)

# P4 quickstart - sentiment: classic vs neural vs zero-shot

Three approaches on the same reviews: TF-IDF + logistic regression, a tiny PyTorch network, and pretrained transformers used zero-shot. CPU is enough; the first run downloads two small Hugging Face models (25 MB and 260 MB). About 5 minutes.

## 1. Get the code
In Colab this clones the repo. If the project folder already exists, the clone is skipped.

In [1]:
import os, subprocess, sys
from pathlib import Path

PROJECT = "15-projects/p04-sentiment-classic-vs-neural"
if Path("main.py").exists() and Path("requirements.txt").exists():
    print("Already inside the project folder - no clone needed.")
elif Path("../../" + PROJECT).exists():
    os.chdir("../../" + PROJECT)          # repo already on disk two levels up
else:
    if not Path("AI-Engineer").exists():  # first run in Colab: download the repo
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/vvvvvivekkk/AI-Engineer.git"], check=True)
    os.chdir(Path("AI-Engineer") / PROJECT)
print("Working folder:", Path.cwd())

Already inside the project folder - no clone needed.
Working folder: /home/claude/AI-Engineer/15-projects/p04-sentiment-classic-vs-neural


## 2. Install packages (Colab only)

In [2]:
import sys, subprocess
from pathlib import Path
# Only install in Colab. Locally you already did "pip install -r requirements.txt" in a venv.
# Colab ships its own PyTorch, so we skip the torch line to save a big download.
if "google.colab" in sys.modules:
    reqs = [l for l in Path("requirements.txt").read_text().splitlines()
            if l.strip() and not l.startswith(("#", "torch"))]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
    print("installed", len(reqs), "packages")
else:
    print("Not in Colab - skipping install.")

Not in Colab - skipping install.


## 3. (Re)create the data
Seeded templates, so the reviews are identical every run.

In [3]:
!{sys.executable} make_data.py
import pandas as pd
pd.read_csv('data/reviews.csv').head()

wrote data/reviews.csv (1000 rows, 47 flipped labels) and data/challenge.csv (40 rows)


,text,label,label_flipped
0,"The customer service is not awful at all, but ...",negative,False
1,"The material is very perfect. Lovely blender, ...",positive,False
2,The build quality is not disappointing at all....,positive,False
3,Sadly the desk lamp is not perfect at all.,negative,False
4,The sound is not flimsy at all.,positive,False


## 4. Train and compare all models

In [4]:
!{sys.executable} main.py

train 800 | test 200 | challenge 40


tfidf_logreg         test acc 0.890 | challenge acc 0.550 | 0.10 ms/review


embedding_bag_nn     test acc 0.835 | challenge acc 0.575 | 0.12 ms/review


Loading weights:   0%|                                 | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 105/105 [00:00<00:00, 2825.35it/s]


zero_shot_nli        test acc 0.895 | challenge acc 0.725 | 271.66 ms/review


Loading weights: 100%|█████████████████████| 104/104 [00:00<00:00, 6398.45it/s]


zero_shot_sst2       test acc 0.945 | challenge acc 0.775 | 191.54 ms/review



written to /home/claude/AI-Engineer/15-projects/p04-sentiment-classic-vs-neural/reports/comparison.md


## 5. Read the comparison and the tricky cases

In [5]:
from IPython.display import Markdown, display
display(Markdown(Path('reports/comparison.md').read_text()))
pd.read_csv('reports/challenge_predictions.csv').head(12)

# Sentiment comparison

Test set: 200 template reviews (8 have deliberately flipped labels, so the best possible accuracy is about 0.960). Challenge set: 40 hand-written tricky reviews.

| model            |   test_accuracy |   test_f1 |   challenge_accuracy |   train_seconds |   ms_per_review |   challenge_share_predicted_positive |   ch_mixed |   ch_negation |   ch_sarcasm |   ch_unseen_words |
|:-----------------|----------------:|----------:|---------------------:|----------------:|----------------:|-------------------------------------:|-----------:|--------------:|-------------:|------------------:|
| tfidf_logreg     |           0.89  |     0.89  |                0.55  |           0.059 |           0.104 |                                0.725 |      0.75  |           0.7 |        0.125 |               0.5 |
| embedding_bag_nn |           0.835 |     0.827 |                0.575 |          32.789 |           0.119 |                                0     |      0.5   |           0.4 |        1     |               0.5 |
| zero_shot_nli    |           0.895 |     0.889 |                0.725 |           0     |         271.663 |                                0.55  |      0.917 |           0.7 |        0.125 |               1   |
| zero_shot_sst2   |           0.945 |     0.944 |                0.775 |           0     |         191.544 |                                0.6   |      0.833 |           0.9 |        0.25  |               1   |

## What the TF-IDF model relies on

- strongest positive signals: awful at, lovely, would buy, all, at all, at, worried, was
- strongest negative signals: sadly, sadly the, not perfect, overall sadly, not reliable, cannot, cannot recommend, awful

The neural model had never seen 65% of the words in the challenge set (vs 0% on the test set).

![accuracy](accuracy.png)


,text,label,kind,tfidf_logreg,embedding_bag_nn,zero_shot_nli,zero_shot_sst2
0,"Oh wonderful, it stopped working after two day...",negative,sarcasm,positive,negative,positive,positive
1,"Great, another charger that melts. Thanks a lot.",negative,sarcasm,positive,negative,positive,positive
2,Fantastic way to waste forty dollars.,negative,sarcasm,positive,negative,positive,negative
3,I just love waiting three weeks for a package ...,negative,sarcasm,positive,negative,negative,negative
4,"Best paperweight I have ever bought, since it ...",negative,sarcasm,positive,negative,positive,positive
5,"Sure, if you enjoy burnt toast every morning, ...",negative,sarcasm,positive,negative,positive,positive
6,"Wow, the strap snapped on day one. Superb engi...",negative,sarcasm,negative,negative,positive,positive
7,Truly amazing how fast the paint peeled off.,negative,sarcasm,positive,negative,positive,positive
8,"Exceeded my expectations, sturdy and elegant.",positive,unseen_words,negative,negative,positive,positive
9,"Brilliant gadget, it works flawlessly.",positive,unseen_words,positive,negative,positive,positive


## 6. Run the tests

In [6]:
!{sys.executable} -m pytest -q

.

.

.

.

.

.                                                                   [100%]
6 passed in 37.43s
